In [1]:
#import

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from dotenv import load_dotenv
import os

#Load the secrets
load_dotenv("secret.env")

MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT")
MINIO_ACCESS_KEY = os.getenv("MINIO_ACCESS_KEY")
MINIO_SECRET_KEY = os.getenv("MINIO_SECRET_KEY")
MINIO_BUCKET_BRONZE = os.getenv("MINIO_BUCKET_BRONZE")

#Spark session creation
spark = SparkSession.builder \
    .appName("Bank_data_pipeline") \
    .master("local[*]") \
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.2"
    ) \
    .config(
        "spark.hadoop.fs.s3a.endpoint",
        f"http://{MINIO_ENDPOINT}"
    ) \
    .config(
        "spark.hadoop.fs.s3a.access.key",
        f"{MINIO_ACCESS_KEY}"
    ) \
    .config(
        "spark.hadoop.fs.s3a.secret.key",
        f"{MINIO_SECRET_KEY}"
    ) \
    .config(
        "spark.hadoop.fs.s3a.path.style.access",
        "true"
    ) \
    .config(
        "spark.hadoop.fs.s3a.connection.ssl.enabled",
        "false"
    ) \
    .getOrCreate()


In [2]:
# Defining the Schema

schema = StructType([
    StructField("step",IntegerType()),
    StructField("type",StringType()),
    StructField("amount",DoubleType()),
    StructField("Account_ID",StringType()),
    StructField("old_balance",DoubleType()),
    StructField("new_balance",DoubleType()),
    StructField("receiver_ID",StringType()),
    StructField("receiver_old_bal",DoubleType()),
    StructField("receiver_new_bal",DoubleType()),
    StructField("is_fraud",IntegerType())
])

cutoff_date = '2026-01-20'

In [3]:
# Reading the data & checking the schema of data

df = spark.read.format("csv")\
               .option("header", True)\
               .schema(schema)\
               .option("mode","PERMISSIVE")\
               .option("path","/home/jupyter/Data_Engineer_project/Dataset/transaction_bank_data.csv")\
               .load()

df.printSchema()

root
 |-- step: integer (nullable = true)
 |-- type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- Account_ID: string (nullable = true)
 |-- old_balance: double (nullable = true)
 |-- new_balance: double (nullable = true)
 |-- receiver_ID: string (nullable = true)
 |-- receiver_old_bal: double (nullable = true)
 |-- receiver_new_bal: double (nullable = true)
 |-- is_fraud: integer (nullable = true)



In [4]:
#Creating Load date by processing the Step column

df_raw = df.withColumn("load_date", date_add(to_date(lit('2026-01-01')), 
                                             (((floor((col("step") - 1) / 24)) + 1) - 1).cast("int")))

In [16]:
# streaming data

df_stream = df_raw.filter(col("load_date")>cutoff_date)\
      .orderBy(col("step"))


df_stream.coalesce(1)\
      .write.mode("overwrite")\
      .format("csv")\
      .option("header",True)\
      .save("/home/jupyter/Data_Engineer_project/transaction_stream")

In [6]:
%pip install kafka-python

Note: you may need to restart the kernel to use updated packages.


In [13]:
import time
from kafka import KafkaProducer
import json

producer = KafkaProducer(
    bootstrap_servers="kafka:29092",
    key_serializer=lambda k: k.encode("utf-8"),
    value_serializer=lambda v: json.dumps(v, default=str).encode("utf-8"),
    acks="all"
)

print("Producer connected")

Producer connected


In [17]:
sent_count = 0

for row in df_stream.orderBy("step").toLocalIterator():

    transaction = row.asDict()

    producer.send(
        "transactions_stream",
        key=str(transaction["Account_ID"]),
        value=transaction
    )

    sent_count += 1

    if sent_count % 1000 == 0:
        print(f"Sent {sent_count} transactions")

    time.sleep(0.05)

producer.flush()

print(f"Finished. Total transactions sent: {sent_count}")

Sent 1000 transactions
Sent 2000 transactions
Sent 3000 transactions
Sent 4000 transactions
Sent 5000 transactions
Sent 6000 transactions
Sent 7000 transactions
Sent 8000 transactions
Sent 9000 transactions
Sent 10000 transactions
Sent 11000 transactions
Sent 12000 transactions
Sent 13000 transactions
Sent 14000 transactions
Sent 15000 transactions
Sent 16000 transactions
Sent 17000 transactions
Sent 18000 transactions
Sent 19000 transactions
Sent 20000 transactions
Sent 21000 transactions
Sent 22000 transactions
Sent 23000 transactions
Sent 24000 transactions
Sent 25000 transactions
Sent 26000 transactions
Sent 27000 transactions
Sent 28000 transactions
Sent 29000 transactions
Sent 30000 transactions
Sent 31000 transactions
Sent 32000 transactions
Sent 33000 transactions
Sent 34000 transactions
Sent 35000 transactions
Sent 36000 transactions
Sent 37000 transactions
Sent 38000 transactions
Sent 39000 transactions
Sent 40000 transactions
Sent 41000 transactions
Sent 42000 transactions
S

Sent 66000 transactions
Sent 67000 transactions
Sent 68000 transactions
Sent 69000 transactions
Sent 70000 transactions
Sent 71000 transactions
Sent 72000 transactions
Sent 73000 transactions
Sent 74000 transactions
Sent 75000 transactions
Sent 76000 transactions
Sent 77000 transactions
Sent 78000 transactions
Sent 79000 transactions
Sent 80000 transactions
Sent 81000 transactions
Sent 82000 transactions
Sent 83000 transactions
Sent 84000 transactions
Sent 85000 transactions
Sent 86000 transactions
Sent 87000 transactions
Sent 88000 transactions
Sent 89000 transactions
Sent 90000 transactions
Sent 91000 transactions
Sent 92000 transactions
Sent 93000 transactions
Sent 94000 transactions
Sent 95000 transactions
Sent 96000 transactions
Sent 97000 transactions
Sent 98000 transactions
Sent 99000 transactions
Sent 100000 transactions
Sent 101000 transactions
Sent 102000 transactions
Sent 103000 transactions
Sent 104000 transactions
Sent 105000 transactions
Sent 106000 transactions
Sent 1070

In [18]:
print("Spark version:", spark.version)

Spark version: 3.3.0
